# NB05 — ablations / permutation null, shard 4 (CPU)

Cell = main-grid rank 1 (chosen on Kaggle from the NB04 outputs). Ablations: `none`. Permutations: 100 starting at 100. Same frozen folds as the grid, so runs are paired.

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
SHA = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
sh(f'pip install -q -e "{SRC}[dev]"', log=f'{WORK}/pip_install.log')
PY = f'cd {SRC} && {sys.executable} -m eegrep'
print('git sha', SHA)

In [ ]:
import glob
CACHE = [d for d in glob.glob('/kaggle/input/**/cache', recursive=True) if os.path.exists(f'{d}/index.npz')][0]
PART = sorted(glob.glob('/kaggle/input/**/participants.tsv', recursive=True))[0]
FOLDS = f'{SRC}/splits/folds_ds004504.json'
print(CACHE, PART, FOLDS)
RUNS = sorted(glob.glob('/kaggle/input/**/runs.jsonl', recursive=True))
print(RUNS); assert len(RUNS) == 3, 'missing grid shard outputs'
CELLS = sh(f'{PY} select-cells --runs {" ".join(RUNS)} --top 2').strip().splitlines()[-1].split(',')
json.dump(CELLS, open(f'{WORK}/cells.json', 'w')); print('ablating', CELLS)

In [ ]:
sh(f'{PY} ablate --cache {CACHE} --folds {FOLDS} --participants {PART} --cells {CELLS[0]} --which none --n-perm 100 --perm-start 100 --out {WORK}/ablations.jsonl --max-hours 10.5', log=f'{WORK}/ablate.log')

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': SHA, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)